# rynude Lyric 4.6 — QLoRA fine-tuning (Qwen3-1.7B) di Google Colab GRATIS

Notebook ini melatih **rynude Lyric 4.6** dari model dasar **rynude Lyric 4.5** (Qwen3-1.7B) memakai QLoRA, lalu mengekspornya ke file **GGUF** yang siap dimasukkan ke Model Hub aplikasi rynude.

**Prasyarat (Rp 0):**
1. Akun Google (untuk Colab).
2. Pilih runtime GPU gratis: menu **Runtime → Change runtime type → T4 GPU → Save**.
3. File `train.jsonl` dan `val.jsonl` hasil `build_dataset.py` (lihat `training/README.md`).

Jalankan sel dari atas ke bawah (Shift+Enter). Total waktu untuk 1.7B: **± 20–60 menit** tergantung ukuran data.

> ⚠️ Colab gratis bisa memutus sesi jika idle. Jangan tutup tab, dan simpan hasil (sel terakhir mengunduh file ke komputer Anda).

## 1. Pasang Unsloth (framework QLoRA tercepat)

In [ ]:
%%capture
# Unsloth: 2x lebih cepat, VRAM 1/2. Versi pip resmi untuk Colab.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade trl peft accelerate bitsandbytes

: 

## 2. Muat model dasar Qwen3-1.7B (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 4096  # cukup untuk 1 bab / dokumen sedang

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-1.7B",   # otomatis versi 4-bit yang pas
    max_seq_length = MAX_SEQ,
    load_in_4bit = True,
    dtype = None,
)
print("Model dasar termuat.")

## 3. Pasang adapter LoRA (hyperparameter dari rancangan loRA.md Bab 6.3)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    lora_alpha = 64,
    lora_dropout = 0.05,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
print("Adapter LoRA terpasang.")

## 4. Unggah data latih

Jalankan sel di bawah, lalu **pilih file `train.jsonl` dan `val.jsonl`** dari komputer Anda (hasil `build_dataset.py`).

In [ ]:
import os
from datasets import load_dataset

# Jika dijalankan di VSCode Colab atau file sudah ada di folder kerja, langsung baca file lokal.
if not os.path.exists("train.jsonl"):
    try:
        from google.colab import files
        print("Pilih/unggah train.jsonl dan val.jsonl ...")
        up = files.upload()
    except Exception as e:
        print("Silakan upload/drag-and-drop file train.jsonl dan val.jsonl ke panel explorer kiri.")

assert os.path.exists("train.jsonl"), "File train.jsonl belum terunggah/ditemukan di folder kerja!"

data_files = {"train": "train.jsonl"}
if os.path.exists("val.jsonl"):
    data_files["validation"] = "val.jsonl"
ds = load_dataset("json", data_files=data_files)

def to_text(ex):
    # Terapkan chat template resmi Qwen3 (WAJIB sama dengan saat inferensi).
    ex["text"] = tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)
    return ex

ds = ds.map(to_text)
print(ds)
print("\nContoh 1 baris terformat:\n", ds["train"][0]["text"][:600])

## 5. Latih (completion-only: loss hanya pada jawaban asisten)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = ds["train"],
    eval_dataset = ds.get("validation"),
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = None,
        max_length = None,
        packing = False,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer.train()


## 6. Uji cepat (pastikan jawaban Bahasa Indonesia & waras)

In [ ]:
FastLanguageModel.for_inference(model)
for q in ["halo bang", "jelaskan apa itu machine learning secara singkat",
          "buatkan surat izin sakit"]:
    msgs = [{"role":"user","content":q}]
    inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                           return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.6,
                         top_p=0.95, top_k=20)
    print("Q:", q)
    print("A:", tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
    print("-"*60)

## 7. Ekspor ke GGUF (file yang dipakai aplikasi rynude)

Menghasilkan **Q8_0** (kualitas terbaik untuk 1.7B, ± 1.8 GB) yang cocok dipakai node-llama-cpp.

In [ ]:
import os
path = None
folders_to_check = ["rynude-lyric-plus_gguf", "rynude-lyric-plus"]
for folder in folders_to_check:
    if os.path.exists(folder):
        for f in os.listdir(folder):
            if f.endswith(".gguf"):
                path = os.path.join(folder, f)
                break
    if path: break

if path:
    print("File GGUF siap di server Colab/VSCode:", path)
    try:
        from google.colab import files
        files.download(path)
    except Exception as e:
        print(f"Jika otomatis unduh tidak aktif di VSCode Colab, silakan klik kanan file '{path}' di panel explorer kiri lalu pilih 'Download'.")
else:
    print("File GGUF tidak ditemukan di folder mana pun!")


## 8. Unduh GGUF ke komputer Anda

In [ ]:
import os
path = None
folders_to_check = ["rynude-lyric-plus_gguf", "rynude-lyric-plus"]
for folder in folders_to_check:
    if os.path.exists(folder):
        for f in os.listdir(folder):
            if f.endswith(".gguf"):
                path = os.path.join(folder, f)
                break
    if path: break

if path:
    print("File GGUF siap di server Colab/VSCode:", path)
    try:
        from google.colab import files
        files.download(path)
    except Exception as e:
        print(f"Jika otomatis unduh tidak aktif di VSCode Colab, silakan klik kanan file '{path}' di panel explorer kiri lalu pilih 'Download'.")
else:
    print("File GGUF tidak ditemukan di folder mana pun!")


## Selesai! Langkah terakhir (di komputer Anda)

1. Rename file GGUF menjadi **`Qwen3-1.7B-Lyric-Plus-Q8_0.gguf`** dan taruh di `storage/app/models/`.
2. Daftarkan sebagai model baru di Model Hub (minta Claude Code menambahkannya ke katalog — jangan menimpa Lyric asli).
3. Ukur hasilnya: `php artisan rynude:eval <kode-model-baru>` dan bandingkan dengan baseline.

Kalau skornya belum naik, **perbaiki DATA** (tambah contoh untuk kasus yang masih gagal), bukan menambah epoch. Kualitas data = kualitas model.